# Funds dosage: intervention-count distributions

This first step shows the observed `InterventionCount` distribution for **(1) the entire Funds causal-forest training set** and **(2) the top 33% of that same training set by causal-forest `benefit_score`**. These are descriptive counts, not dosage effects or recommendations. The benefit score is the existing binary-treatment causal-forest estimate; intervention count was not a predictor in that model.

Run from the repository root or `Code` folder after the Funds causal-forest workflow has produced its scored outputs. No files are written by this notebook.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CODE_DIR = Path.cwd() if (Path.cwd() / '_prism_model_utils.py').exists() else Path.cwd() / 'Code'
if not (CODE_DIR / '_prism_model_utils.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root or Code folder.')
sys.path.insert(0, str(CODE_DIR))
from _prism_model_utils import clean_names_simple, split_train_test, to_binary

ROOT = CODE_DIR.parent
RAW_PATH = ROOT / 'DataSets' / 'Funds_combined.csv'
SCORED_DIR = ROOT / 'Outputs' / 'Causal-Forests_Funds' / 'Python'
FULL_PATH = SCORED_DIR / 'causal_forest_scored_output.csv'
TEST_PATH = SCORED_DIR / 'causal_forest_scored_test_output.csv'
for path in (RAW_PATH, FULL_PATH, TEST_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Required Funds input is missing: {path}')
print('Raw data:', RAW_PATH)
print('Causal-forest scores:', FULL_PATH)

## Reconstruct the exact Funds training cohort

The original workflow uses a 70/30 split stratified by binary treatment and binary 90-day ED outcome, with seed 123. We reproduce it and verify the resulting test row IDs against the saved test output. Because source `member_id` values repeat, the count is aligned by validated source row order and `analysis_row_id`, not by member ID alone.

In [ ]:
raw = pd.read_csv(RAW_PATH)
raw.columns = clean_names_simple(raw.columns)
scored = pd.read_csv(FULL_PATH)
saved_test = pd.read_csv(TEST_PATH)

required_raw = {'member_id', 'intervention_flag', 'optout_flag', 'outcome_ed_90d', 'interventioncount'}
required_scored = {'analysis_row_id', 'member_id', 'intervention_flag', 'outcome_ed_90d', 'benefit_score'}
assert required_raw.issubset(raw), required_raw - set(raw)
assert required_scored.issubset(scored), required_scored - set(scored)

# Mirror the source workflow's eligibility rule, without refitting the model.
source_treatment = to_binary(raw['intervention_flag']).where(lambda s: s.isin([0.0, 1.0]))
source_optout = to_binary(raw['optout_flag']).where(lambda s: s.isin([0.0, 1.0]))
treated = source_treatment.eq(1.0) & ~source_optout.eq(1.0)
control = source_optout.eq(1.0) & ~source_treatment.eq(1.0)
eligible = treated | control
raw_model = raw.loc[eligible].reset_index(drop=True)
expected_treatment = np.select([treated, control], [1.0, 0.0], default=np.nan)[eligible.to_numpy()]
expected_outcome = (pd.to_numeric(raw['outcome_ed_90d'], errors='coerce').fillna(0) > 0).astype(float)[eligible].to_numpy()

assert len(raw_model) == len(scored), 'Source data and scored file have different model-cohort sizes.'
assert scored['analysis_row_id'].tolist() == list(range(len(scored))), 'Scored source-row order changed.'
assert raw_model['member_id'].astype(str).tolist() == scored['member_id'].astype(str).tolist(), 'Source/scored member order differs.'
assert np.array_equal(expected_treatment, scored['intervention_flag'].to_numpy()), 'Treatment alignment failed.'
assert np.array_equal(expected_outcome, scored['outcome_ed_90d'].to_numpy()), 'Outcome alignment failed.'
assert scored['benefit_score'].notna().all(), 'Some modeled rows lack benefit scores.'

scored['intervention_count'] = pd.to_numeric(raw_model['interventioncount'], errors='coerce').to_numpy()
scored['engagement_length'] = pd.to_numeric(
    raw_model['engagement_length'].replace({'NA': np.nan}), errors='coerce'
).to_numpy()
valid_counts = scored['intervention_count'].dropna()
assert (valid_counts >= 0).all() and (valid_counts % 1 == 0).all(), 'Expected nonnegative integer intervention counts.'

train, reconstructed_test = split_train_test(
    scored, train_fraction=0.70, seed=123,
    stratify_columns=['intervention_flag', 'outcome_ed_90d'],
)
assert reconstructed_test['analysis_row_id'].tolist() == saved_test['analysis_row_id'].tolist(), 'Reconstructed split does not match saved Funds test rows.'
assert len(train) + len(reconstructed_test) == len(scored)
print(f'Exact split verified: {len(train):,} training rows and {len(reconstructed_test):,} test rows.')

## Group 1: entire training set

Missing count values have their own row; they are **not** recoded to zero. Percentages use all rows in the named group as the denominator. Summary statistics use only non-missing counts. `Range width` is maximum minus minimum.

In [ ]:
def count_distribution(frame):
    counts = frame['intervention_count'].value_counts(dropna=False).sort_index()
    labels = ['Missing' if pd.isna(value) else str(int(value)) for value in counts.index]
    return pd.DataFrame({
        'intervention_count': labels,
        'people': counts.to_numpy(dtype=int),
        'percent_of_group': (100 * counts.to_numpy() / len(frame)).round(2),
    })

def count_summary(frame):
    observed = frame['intervention_count'].dropna()
    minimum = observed.min() if not observed.empty else np.nan
    maximum = observed.max() if not observed.empty else np.nan
    return pd.DataFrame([{
        'total_people': len(frame),
        'nonmissing_counts': len(observed),
        'missing_counts': len(frame) - len(observed),
        'mean': round(observed.mean(), 2),
        'minimum': minimum,
        'maximum': maximum,
        'observed_range': f'{int(minimum)}–{int(maximum)}' if not observed.empty else 'N/A',
        'range_width': maximum - minimum,
    }])

def plot_distribution(distribution, title):
    fig, ax = plt.subplots(figsize=(max(12, 0.27 * len(distribution)), 5))
    ax.bar(distribution['intervention_count'], distribution['people'], color='#2673a8')
    ax.set(title=title, xlabel='Observed intervention count', ylabel='Number of people')
    ax.tick_params(axis='x', labelrotation=90)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()

full_train_dist = count_distribution(train)
print(f'Entire Funds training set: {len(train):,} rows; {train.intervention_count.isna().sum():,} missing intervention counts.')
display(count_summary(train))
display(full_train_dist)
plot_distribution(full_train_dist, 'Funds training set: intervention-count distribution')

## Group 2: highest-benefit 33% of the training set

Rank only training rows by the existing causal-forest `benefit_score` (larger means greater predicted ED reduction under binary treatment). Select `ceil(0.33 × training rows)`; use `analysis_row_id` to break exact score ties reproducibly. This is a subset of Group 1, not a separately fitted dosage model.

In [ ]:
top_n = int(np.ceil(0.33 * len(train)))
high_benefit = train.sort_values(
    ['benefit_score', 'analysis_row_id'], ascending=[False, True], kind='stable'
).head(top_n).copy()
assert high_benefit['analysis_row_id'].isin(train['analysis_row_id']).all()
assert len(high_benefit) == top_n
high_benefit_dist = count_distribution(high_benefit)
print(f'Highest-benefit training group: {len(high_benefit):,} of {len(train):,} rows ({100 * len(high_benefit) / len(train):.2f}%); {high_benefit.intervention_count.isna().sum():,} missing intervention counts.')
display(count_summary(high_benefit))
display(high_benefit_dist)
plot_distribution(high_benefit_dist, 'Top 33% by Funds causal-forest benefit: intervention-count distribution')

## Engagement-length distribution and summary statistics

This section uses the same two training groups as the intervention-count analysis. Engagement length is treated as days. The buckets are `0`, `1–30`, `31–60`, `61–90`, `91–180`, `181–365`, and `>365` days. Missing values and negative values are shown separately; negative values are treated as invalid for summary statistics rather than silently included.

In [ ]:
ENGAGEMENT_BUCKETS = ['Invalid (<0)', '0 days', '1–30 days', '31–60 days', '61–90 days', '91–180 days', '181–365 days', '>365 days', 'Missing']

def engagement_length_distribution(frame):
    values = pd.to_numeric(frame['engagement_length'], errors='coerce')
    bucket = pd.Series(pd.NA, index=frame.index, dtype='object')
    bucket.loc[values.isna()] = 'Missing'
    bucket.loc[values.lt(0)] = 'Invalid (<0)'
    bucket.loc[values.eq(0)] = '0 days'
    bucket.loc[values.gt(0) & values.le(30)] = '1–30 days'
    bucket.loc[values.gt(30) & values.le(60)] = '31–60 days'
    bucket.loc[values.gt(60) & values.le(90)] = '61–90 days'
    bucket.loc[values.gt(90) & values.le(180)] = '91–180 days'
    bucket.loc[values.gt(180) & values.le(365)] = '181–365 days'
    bucket.loc[values.gt(365)] = '>365 days'
    counts = bucket.value_counts().reindex(ENGAGEMENT_BUCKETS, fill_value=0)
    return pd.DataFrame({'engagement_length_bucket': counts.index, 'people': counts.to_numpy(dtype=int), 'percent_of_group': (100 * counts.to_numpy() / len(frame)).round(2)})

def engagement_length_summary(frame):
    values = pd.to_numeric(frame['engagement_length'], errors='coerce')
    observed = values.loc[values.ge(0)].dropna()
    minimum = observed.min() if not observed.empty else np.nan
    maximum = observed.max() if not observed.empty else np.nan
    return pd.DataFrame([{
        'total_people': len(frame),
        'nonmissing_values': int(values.notna().sum()),
        'missing_values': int(values.isna().sum()),
        'invalid_negative_values': int(values.lt(0).sum()),
        'valid_observed_values': len(observed),
        'mean_days': round(observed.mean(), 2) if not observed.empty else np.nan,
        'median_days': round(observed.median(), 2) if not observed.empty else np.nan,
        'minimum_days': minimum,
        'maximum_days': maximum,
        'observed_range_days': f'{minimum:g}–{maximum:g}' if not observed.empty else 'N/A',
        'range_width_days': maximum - minimum if not observed.empty else np.nan,
    }])

def plot_engagement_length_distribution(distribution, title):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(distribution['engagement_length_bucket'], distribution['people'], color='#4c956c')
    ax.set(title=title, xlabel='Engagement length bucket', ylabel='Number of people')
    ax.tick_params(axis='x', labelrotation=35)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()

for group_name, group_frame in [('Entire Funds training set', train), ('Highest-benefit training group', high_benefit)]:
    distribution = engagement_length_distribution(group_frame)
    print(f'{group_name}: {len(group_frame):,} rows')
    display(engagement_length_summary(group_frame))
    display(distribution)
    plot_engagement_length_distribution(distribution, f'{group_name}: engagement-length distribution')